In [ ]:
# import json

# with open("2.MARCIA_result_pages.json", encoding="utf-8") as f:
#     datos = json.load(f)

# base_url = "https://marcia.impi.gob.mx/marcas/search/details/"

# urls = [
#     base_url + item["id"]
#     for item in datos
#     if item.get("id")
# ]

# print(len(urls))
# print(urls[:5])  # muestra las primeras

153978
['https://marcia.impi.gob.mx/marcas/search/details/AC202000129767', 'https://marcia.impi.gob.mx/marcas/search/details/AC202100134698', 'https://marcia.impi.gob.mx/marcas/search/details/AC202100134859', 'https://marcia.impi.gob.mx/marcas/search/details/AC202100135352', 'https://marcia.impi.gob.mx/marcas/search/details/AC202100135353']


In [ ]:
# import json
# import time
# import random
# import requests
# from pathlib import Path
# from tqdm import tqdm
# from bs4 import XMLParsedAsHTMLWarning
# import warnings
# import urllib3

# # ============================== CONFIGURACIÓN ==============================
# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# BASE_URL      = "https://marcia.impi.gob.mx"
# ORIGINAL_JSON = Path("2.MARCIA_result_pages.json")

# # ==================== ARCHIVOS DE SALIDA ====================
# RESULTADOS_JSONL = Path("4.marcia_details.jsonl")
# PROCESSED_FILE   = Path("4.processed_ids.txt")
# FALLIDOS_JSONL   = Path("4.fallidos.jsonl")

# # ==================== PARÁMETROS ====================
# BATCH_SIZE        = 99
# SLEEP_PER_RECORD  = random.uniform(0.01, 0.05)
# SLEEP_AFTER_BATCH = 0.2
# MAX_RETRIES       = 2
# BACKOFF_FACTOR    = 2


# # ======================== SESIÓN ========================
# def crear_sesion():
#     session = requests.Session()
#     session.get(f"{BASE_URL}/marcas/search/quick")
#     xsrf_token = session.cookies.get("XSRF-TOKEN")
#     headers = {
#         "X-XSRF-TOKEN": xsrf_token,
#         "Referer":       f"{BASE_URL}/marcas/search/quick"
#     }
#     return session, headers


# # ======================== FUNCIÓN PRINCIPAL ========================
# def obtener_detalle(id_marca: str, session, headers) -> dict:
#     url = f"{BASE_URL}/marcas/search/internal/view/{id_marca}?sort=&pageSize=100"
#     r   = session.get(url, headers=headers, timeout=20)
#     r.raise_for_status()
#     datos = r.json()

#     pdf_links = [
#         rec.get("image", "")
#         for rec in datos.get("historyData", {}).get("historyRecords", [])
#         if rec.get("image")
#     ]

#     return {
#         "id":          id_marca,
#         "details":     datos.get("details", {}),
#         "historyData": datos.get("historyData", {}),
#         "pdf_links":   pdf_links
#     }


# # ======================== SISTEMA DE PROGRESO ========================
# def cargar_set(filepath: Path) -> set:
#     if filepath.exists():
#         return {line.strip() for line in filepath.read_text(encoding="utf-8").splitlines() if line.strip()}
#     return set()


# def guardar_set(data: set, filepath: Path):
#     filepath.write_text("\n".join(sorted(data)) + "\n", encoding="utf-8")


# def append_jsonl(data: dict, filepath: Path):
#     with filepath.open("a", encoding="utf-8") as f:
#         f.write(json.dumps(data, ensure_ascii=False) + "\n")


# def cargar_fallidos_set() -> set:
#     if not FALLIDOS_JSONL.exists():
#         return set()
#     fallidos = set()
#     with FALLIDOS_JSONL.open(encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             try:
#                 obj = json.loads(line)
#                 if obj.get("id"):
#                     fallidos.add(str(obj["id"]))
#             except json.JSONDecodeError:
#                 continue
#     return fallidos


# # ← NUEVO: sincronizar processed_ids con lo que ya está en el jsonl
# def sincronizar_processed(processed: set) -> set:
#     """
#     Lee 4.marcia_details.jsonl y agrega al set de processed
#     cualquier id que ya esté guardado pero no en el .txt.
#     Evita duplicados si se pausa y reanuda a mitad de un batch.
#     """
#     if not RESULTADOS_JSONL.exists():
#         return processed

#     ids_en_jsonl = set()
#     with RESULTADOS_JSONL.open(encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             try:
#                 obj = json.loads(line)
#                 if obj.get("id") and "error" not in obj:
#                     ids_en_jsonl.add(obj["id"])
#             except json.JSONDecodeError:
#                 continue

#     nuevos = ids_en_jsonl - processed
#     if nuevos:
#         print(f"  → Sincronizando {len(nuevos):,} IDs del jsonl que no estaban en processed_ids.txt")
#         processed.update(nuevos)
#         guardar_set(processed, PROCESSED_FILE)

#     return processed


# # ======================== EJECUCIÓN ========================
# def ejecutar_scraping():
#     print("Iniciando scraper MARCIA details...\n")

#     with ORIGINAL_JSON.open(encoding="utf-8") as f:
#         datos = json.load(f)

#     todos_ids = [item["id"] for item in datos if item.get("id")]

#     processed = cargar_set(PROCESSED_FILE)
#     processed = sincronizar_processed(processed)   # ← NUEVO: sincronizar antes de empezar

#     fallidos             = cargar_fallidos_set()
#     procesados_al_inicio = len(processed)
#     bytes_totales        = 0

#     pending = [
#         id_marca for id_marca in todos_ids
#         if id_marca not in processed and id_marca not in fallidos
#     ]

#     print(f"Total IDs originales  : {len(todos_ids):,}")
#     print(f"Ya procesados         : {len(processed):,}")
#     print(f"Fallidos definitivos  : {len(fallidos):,}")
#     print(f"Pendientes            : {len(pending):,}\n")

#     if not pending:
#         print("Todos los IDs ya fueron procesados.")
#         return

#     session, headers = crear_sesion()
#     start_time_total  = time.time()

#     for i in range(0, len(pending), BATCH_SIZE):
#         batch_start = time.time()
#         batch       = pending[i:i + BATCH_SIZE]
#         print(f"\nBatch {i//BATCH_SIZE + 1}/{-(-len(pending)//BATCH_SIZE)} ({len(batch)} registros)")

#         for id_marca in tqdm(batch, desc="IDs en este batch", leave=False):

#             for intento in range(1, MAX_RETRIES + 1):
#                 try:
#                     detalle = obtener_detalle(id_marca, session, headers)

#                     if not detalle.get("details"):
#                         raise Exception("Respuesta vacía")

#                     append_jsonl(detalle, RESULTADOS_JSONL)
#                     processed.add(id_marca)
#                     bytes_totales += len(json.dumps(detalle).encode("utf-8"))

#                     time.sleep(SLEEP_PER_RECORD + random.uniform(0.2, 0.8))
#                     break

#                 except Exception as e:
#                     if intento == MAX_RETRIES:
#                         print(f"  ✗ Falló: {id_marca} → {e}")
#                         append_jsonl({"id": id_marca, "error": str(e)}, FALLIDOS_JSONL)
#                         fallidos.add(id_marca)

#                         if "connection" in str(e).lower() or "timeout" in str(e).lower():
#                             print("  → Renovando sesión...")
#                             session, headers = crear_sesion()
#                     else:
#                         time.sleep(BACKOFF_FACTOR ** (intento - 1))

#         # ── estadísticas ────────────────────────────────────────
#         batch_time        = time.time() - batch_start
#         total_time        = time.time() - start_time_total
#         nuevos            = len(processed) - procesados_al_inicio
#         velocidad         = nuevos / total_time if total_time > 0 else 0
#         pendientes_reales = len(pending) - nuevos
#         estimado_min      = (pendientes_reales / velocidad / 60) if velocidad > 0 else 0

#         if nuevos > 0:
#             mb_restantes = (pendientes_reales * (bytes_totales / nuevos)) / (1024 ** 2)
#         else:
#             mb_restantes = 0

#         print(f"Batch: {batch_time/60:.1f} min | Velocidad: {velocidad:.1f} reg/min")
#         print(f"Nuevos esta sesión: {nuevos:,} | Estimado: {estimado_min:.0f} min ({estimado_min/60:.1f} hrs)")
#         print(f"Tráfico acumulado: {bytes_totales/1024**2:.1f} MB | Restante estimado: {mb_restantes:.0f} MB")

#         guardar_set(processed, PROCESSED_FILE)

#         print(f"Pausa de {SLEEP_AFTER_BATCH:.0f}s...")
#         time.sleep(SLEEP_AFTER_BATCH)

#     print("\nScraping completado.")


# if __name__ == "__main__":
#     try:
#         ejecutar_scraping()
#     except KeyboardInterrupt:
#         print("\n\nInterrumpido. Progreso guardado.")
#         guardar_set(cargar_set(PROCESSED_FILE), PROCESSED_FILE)

Iniciando scraper MARCIA details...

Total IDs originales  : 153,978
Ya procesados         : 153,977
Fallidos definitivos  : 0
Pendientes            : 1


Batch 1/1 (1 registros)


Batch: 0.0 min | Velocidad: 1.1 reg/min
Nuevos esta sesión: 1 | Estimado: 0 min (0.0 hrs)
Tráfico acumulado: 0.0 MB | Restante estimado: 0 MB
Pausa de 0s...

Scraping completado.


In [5]:
import json
import pandas as pd

#data = json.load(open("4.marcia_details.jsonl", encoding="utf-8"))

with open("4.marcia_details.jsonl", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]